In [ ]:
"""
第1部分：Elasticsearch环境准备和索引创建
"""

from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
import json

# ==================== 1. 连接Elasticsearch ====================
def connect_elasticsearch(host='localhost', port=9200):
    """连接ES服务器"""
    try:
        es = Elasticsearch([{'host': host, 'port': port, 'scheme': 'http'}])
        # 测试连接
        if es.ping():
            print("✅ 成功连接到Elasticsearch")
            print(f"ES版本: {es.info()['version']['number']}")
            return es
        else:
            print("❌ 无法连接到Elasticsearch")
            return None
    except Exception as e:
        print(f"❌ 连接失败: {e}")
        return None


# ==================== 2. 创建索引映射 ====================
def create_index_mapping():
    """定义索引的mapping（字段类型和分析器）"""
    
    mapping = {
        "settings": {
            "number_of_shards": 1,
            "number_of_replicas": 0,
            "analysis": {
                "analyzer": {
                    # 中文分词器（使用IK分词）
                    "ik_smart_analyzer": {
                        "type": "custom",
                        "tokenizer": "ik_smart"
                    },
                    "ik_max_word_analyzer": {
                        "type": "custom",
                        "tokenizer": "ik_max_word"
                    }
                }
            }
        },
        "mappings": {
            "properties": {
                # 基础信息
                "rule_id": {"type": "keyword"},
                "disease": {"type": "keyword"},
                "phase": {"type": "keyword"},
                "treatment_type": {"type": "keyword"},
                "population": {"type": "keyword"},
                "language": {"type": "keyword"},
                
                # 干预信息
                "intervention": {
                    "properties": {
                        "name": {
                            "type": "text",
                            "analyzer": "ik_max_word",
                            "search_analyzer": "ik_smart",
                            "fields": {
                                "keyword": {"type": "keyword"}
                            }
                        },
                        "category": {"type": "keyword"},
                        "sub_category": {"type": "keyword"},
                        "route": {
                            "properties": {
                                "primary": {"type": "keyword"},
                                "description": {
                                    "type": "text",
                                    "analyzer": "ik_smart"
                                }
                            }
                        },
                        "frequency": {
                            "type": "text",
                            "analyzer": "ik_smart"
                        },
                        "course": {"type": "text"},
                        "constraints": {"type": "text"}
                    }
                },
                
                # 适用性
                "applicability": {
                    "properties": {
                        "age_range": {"type": "keyword"},
                        "pregnancy_lactation": {"type": "keyword"},
                        "comorbidities_excluded": {"type": "keyword"}
                    }
                },
                
                # 证据
                "evidence": {
                    "properties": {
                        "evidence_level": {"type": "keyword"},
                        "recommendation_grade": {"type": "keyword"},
                        "evidence_source": {"type": "keyword"},
                        "study_count": {"type": "integer"},
                        "consistency": {"type": "keyword"}
                    }
                },
                
                # 安全性
                "safety": {
                    "properties": {
                        "adverse_effects": {
                            "properties": {
                                "common": {"type": "text", "analyzer": "ik_smart"},
                                "rare": {"type": "text"},
                                "severity": {"type": "keyword"}
                            }
                        },
                        "contraindications": {"type": "text", "analyzer": "ik_smart"},
                        "precautions": {"type": "text"}
                    }
                },
                
                # 来源
                "source": {
                    "properties": {
                        "guideline": {
                            "type": "text",
                            "analyzer": "ik_smart",
                            "fields": {
                                "keyword": {"type": "keyword"}
                            }
                        },
                        "version": {"type": "keyword"},
                        "year": {"type": "integer"},
                        "section": {"type": "text"},
                        "table_title": {"type": "text"},
                        "pdf_id": {"type": "keyword"},
                        "page": {"type": "integer"},
                        "source_type": {"type": "keyword"}
                    }
                },
                
                # 可追溯性
                "traceability": {
                    "properties": {
                        "table_row_index": {"type": "integer"},
                        "footnote_refs": {"type": "text"},
                        "raw_text_segment": {"type": "text"}
                    }
                },
                
                "extraction_method": {"type": "keyword"}
            }
        }
    }
    
    return mapping


# ==================== 3. 创建索引 ====================
def create_index(es, index_name='medical_guidelines'):
    """创建ES索引"""
    
    # 如果索引已存在，先删除
    if es.indices.exists(index=index_name):
        print(f"⚠️  索引 '{index_name}' 已存在，正在删除...")
        es.indices.delete(index=index_name)
    
    # 创建新索引
    mapping = create_index_mapping()
    es.indices.create(index=index_name, body=mapping)
    print(f"✅ 成功创建索引 '{index_name}'")


# ==================== 4. 批量导入数据 ====================
def bulk_import_data(es, index_name, json_file):
    """批量导入JSON数据到ES"""
    
    # 读取JSON文件
    with open(json_file, 'r', encoding='utf-8') as f:
        documents = json.load(f)
    
    print(f"📄 共读取 {len(documents)} 条文档")
    
    # 准备批量导入的actions
    actions = []
    for doc in documents:
        action = {
            "_index": index_name,
            "_id": doc['rule_id'],  # 使用rule_id作为文档ID
            "_source": doc
        }
        actions.append(action)
    
    # 批量导入
    success, failed = bulk(es, actions, stats_only=True)
    print(f"✅ 成功导入 {success} 条")
    if failed > 0:
        print(f"❌ 失败 {failed} 条")
    
    # 刷新索引
    es.indices.refresh(index=index_name)
    print(f"✅ 索引已刷新")
    
    return success, failed


# ==================== 5. 验证数据 ====================
def verify_import(es, index_name):
    """验证导入的数据"""
    
    # 统计总数
    count = es.count(index=index_name)['count']
    print(f"\n📊 索引中共有 {count} 条文档")
    
    # 按治疗类型统计
    agg_query = {
        "size": 0,
        "aggs": {
            "by_treatment_type": {
                "terms": {"field": "treatment_type", "size": 10}
            },
            "by_phase": {
                "terms": {"field": "phase", "size": 10}
            },
            "by_category": {
                "terms": {"field": "intervention.category", "size": 20}
            }
        }
    }
    
    result = es.search(index=index_name, body=agg_query)
    
    print("\n按治疗类型统计:")
    for bucket in result['aggregations']['by_treatment_type']['buckets']:
        print(f"  - {bucket['key']}: {bucket['doc_count']} 条")
    
    print("\n按治疗阶段统计:")
    for bucket in result['aggregations']['by_phase']['buckets']:
        print(f"  - {bucket['key']}: {bucket['doc_count']} 条")
    
    print("\n按药物分类统计 (Top 10):")
    for bucket in result['aggregations']['by_category']['buckets'][:10]:
        print(f"  - {bucket['key']}: {bucket['doc_count']} 条")


# ==================== 主函数 ====================
if __name__ == "__main__":
    # 配置
    INDEX_NAME = 'medical_guidelines'
    JSON_FILE = 'all_elasticsearch_docs.json'
    
    # 步骤1: 连接ES
    es = connect_elasticsearch()
    if not es:
        exit(1)
    
    # 步骤2: 创建索引
    create_index(es, INDEX_NAME)
    
    # 步骤3: 导入数据
    success, failed = bulk_import_data(es, INDEX_NAME, JSON_FILE)
    
    # 步骤4: 验证数据
    verify_import(es, INDEX_NAME)
    
    print("\n" + "="*60)
    print("✅ 数据导入完成！")
    print("="*60)